# Manage Discovery scheduled scans

Read and update scheduled Discovery runs using `/discovery/runs/scheduled` (GET, POST, PATCH, and DELETE).

The notebook is read-only by default. Set `APPLY = True` only after reviewing the configured mutations.

## Setup

`PATCHES` updates existing schedules, `NEW_SCHEDULE` creates one, and `DELETE_IDS` removes schedules. Leave any operation empty when it is not needed. Request bodies must match the schedule schema returned by the appliance.

In [ ]:
from pprint import pprint

import pandas as pd
from tideway import notebooks as tw_nb

APPLIANCE_NAME = None
APPLIANCE_INDEX = 0
OUTPUT_BASE_DIR = None

tw = tw_nb.appliance_from_config(
    appliance_name=APPLIANCE_NAME,
    appliance_index=APPLIANCE_INDEX,
)
discovery = tw.discovery()
output_dir = tw_nb.output_dir_for(tw.target, OUTPUT_BASE_DIR)
print('Target:', tw.target)
print('Output dir:', output_dir)

# Safety switch: GET remains available with APPLY=False; mutations are skipped.
APPLY = False

# PATCH existing schedules. Each item needs the schedule id and a JSON body.
# Example: {'id': 'run-id', 'body': {'schedule': {'start_times': [2, 14]}}}
PATCHES = []

# POST a new scheduled scan. Leave empty to skip creation.
# Example shape: {'label': 'Nightly scan', 'schedule': {...}, ...}
NEW_SCHEDULE = {}

# DELETE existing schedules by id. Leave empty to skip deletion.
DELETE_IDS = []


## GET: inspect scheduled scans

In [ ]:
def response_json(response):
    if not getattr(response, 'ok', True):
        raise RuntimeError(
            f'HTTP {getattr(response, "status_code", "?")}: {getattr(response, "text", "")}'
        )
    return response.json()


def schedule_rows(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ('results', 'schedules', 'runs'):
            if isinstance(payload.get(key), list):
                return payload[key]
        return [payload]
    return []

get_response = discovery.get_discovery_run_schedules
schedules_payload = response_json(get_response)
schedules = schedule_rows(schedules_payload)
print(f'Found {len(schedules)} scheduled scan(s).')
if schedules:
    display(pd.json_normalize(schedules))
else:
    pprint(schedules_payload)


## POST, PATCH, and DELETE

Mutations run only when `APPLY` is `True`. Afterward the notebook performs another GET so the appliance state can be checked.

In [ ]:
if not APPLY:
    print('No changes made. Set APPLY = True after reviewing PATCHES, NEW_SCHEDULE, and DELETE_IDS.')
else:
    if NEW_SCHEDULE:
        created = discovery.post_discovery_run_schedule(NEW_SCHEDULE)
        print('POST /discovery/runs/scheduled:', created.status_code, created.text)

    for item in PATCHES:
        run_id = item.get('id')
        body = item.get('body')
        if not run_id or not isinstance(body, dict):
            raise ValueError("Each PATCH entry must contain a non-empty 'id' and a dict 'body'.")
        updated = discovery.patch_discovery_run_schedule(run_id, body)
        print(f'PATCH /discovery/runs/scheduled/{run_id}:', updated.status_code, updated.text)

    for run_id in DELETE_IDS:
        deleted = discovery.delete_discovery_run_schedule(run_id)
        print(f'DELETE /discovery/runs/scheduled/{run_id}:', deleted.status_code, deleted.text)

    final_response = discovery.get_discovery_run_schedules
    final_payload = response_json(final_response)
    final_schedules = schedule_rows(final_payload)
    print(f'Final scheduled scan count: {len(final_schedules)}')
    if final_schedules:
        display(pd.json_normalize(final_schedules))
    else:
        pprint(final_payload)
